# Hate Speech Detection

In [1]:
!pip install transformers datasets torch scikit-learn

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.utils.class_weight import compute_class_weight
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import re
from transformers import AutoTokenizer
from torch.utils.data import Dataset

# Training
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

# Results
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import seaborn as sns

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Datasets

In [4]:
def load_dataset_splits(train_path, val_path, test_path):
    train = pd.read_csv(train_path)
    val = pd.read_csv(val_path)
    test = pd.read_csv(test_path)
    return train, val, test

In [5]:
def split_dataset(df):
    """
        Split a given dataset into train, test and validation sets, with
        80% train, 10% test and 10% validation
    """
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

    # Split the 20% into 10% validation and 10% test
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

    return train_df, val_df, test_df

In [6]:
def remove_failed_translations(df_train, df_val, df_test, col_name):
    # Train
    rows_train = df_train.shape[0]
    df_train = df_train.dropna(subset=[col_name])
    print("Nan values found in train set: ", rows_train - df_train.shape[0])

    # Validation
    rows_val = df_val.shape[0]
    df_val = df_val.dropna(subset=[col_name])
    print("Nan values found in validation set: ", rows_val - df_val.shape[0])

    # Test
    rows_test = df_test.shape[0]
    df_test = df_test.dropna(subset=[col_name])
    print("Nan values found in test set: ", rows_test - df_test.shape[0])

    return df_train, df_val, df_test

In [7]:
def drop_columns(train, val, test, cols, label_col):
    if cols != None:
        train = train.drop(columns=cols)
        val = val.drop(columns=cols)
        test = test.drop(columns=cols)

    train["labels"] = train[label_col]
    val["labels"] = val[label_col]
    test["labels"] = test[label_col]

    train = train.drop(columns=[label_col])
    val = val.drop(columns=[label_col])
    test = test.drop(columns=[label_col])

    return train, val, test

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Preprocessing

In [9]:
# Compute class weights
def get_class_weights(train_labels, classes, device):
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=train_labels
    )
    return torch.tensor(class_weights, dtype=torch.float).to(device)

def compute_class_weights(train_df, classes, device):
    # Get weights for loss function
    train_labels = train_df['labels'].values
    class_weights = get_class_weights(train_labels, classes, device)

    # Define weighted loss function
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    return loss_fn

In [10]:
# Download necessary NLTK data for preprocessing
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
def clean_text_translated(text):
    print("Preprocessing translated")
    """Preprocess a single text instance."""
    if pd.isna(text):
        return ""  # Handle missing values

    text = text.lower()  # Lowercase all text

    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags

    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs

    text = re.sub(r"[^\x00-\x7F]+", "", text)  # Remove non-ASCII characters

    text = re.sub(r"([a-z])\1{2,}", r"\1", text)  # Reduce repeated letters (e.g., "gooood" → "good")

    text = re.sub(r"[^\w\s]", " ", text)  # Remove excessive punctuation

    text = re.sub(r"\s+", " ", text).strip()  # Remove extra whitespace

    tokens = word_tokenize(text)  # Tokenization


    return " ".join(tokens)  # Reconstruct cleaned text

def clean_text_original(text):
    print("Preprocessing Original")
    """Preprocess a single text instance."""
    if pd.isna(text):
        return ""  # Handle missing values

    if lang != "ch":
        text = text.lower()  # Lowercase all text

    # For all languages
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs

    if lang == "it" or lang == "nl":
        text = re.sub(r"[^\x00-\x7F]+", "", text)  # Remove non-ASCII characters
        text = re.sub(r"([a-z])\1{2,}", r"\1", text)  # Reduce repeated letters
        text = re.sub(r"[^\w\s]", " ", text)  # Remove excessive punctuation

    if lang == "bg":
        text = re.sub(r"([а-яА-Я])\1{2,}", r"\1", text)
        text = re.sub(r"[^\w\s]", " ", text)  # Remove excessive punctuation

    if lang == "ru":
        text = re.sub(r"([a-zа-яА-Я])\1{2,}", r"\1", text)
        text = re.sub(r"[^\w\sа-яА-ЯёЁ]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()  # Remove extra whitespace

    tokens = word_tokenize(text)  # Tokenization

    return " ".join(tokens)  # Reconstruct cleaned text

def preprocess_dataframe(df, text_col, original):
    """Apply preprocessing to an entire dataset."""

    if self.original:
        df[text_col] = df[text_col].astype(str).apply(clean_text_original)
    else:
        df[text_col] = df[text_col].astype(str).apply(clean_text_translated)
    return df

In [12]:
def preprocess_dataset(train, val, test, textCol, original):
    train = preprocess_dataframe(train, textCol, original)
    val = preprocess_dataframe(val, textCol, original)
    test = preprocess_dataframe(test, textCol, original)
    return train, val, test

In [13]:
def get_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    return tokenizer

# Now use the Hugging Face tokenizer for the final tokenization
def tokenize_texts(texts, tokenizer):
    return tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

def tokenization(train, val, test, labelCol, textCol, tokenizer):
    # Ensure 'translated_text' column is converted to a list of strings
    train_texts = train[textCol].astype(str).tolist()  # Convert to list of strings
    val_texts = val[textCol].astype(str).tolist()      # Convert to list of strings
    test_texts = test[textCol].astype(str).tolist()    # Convert to list of strings

    # Tokenize the preprocessed data
    train_encodings = tokenize_texts(train_texts, tokenizer)
    val_encodings = tokenize_texts(val_texts, tokenizer)
    test_encodings = tokenize_texts(test_texts, tokenizer)

    # Convert labels to the format expected by the model
    train_encodings[labelCol] = torch.tensor(pd.Series(train[labelCol]).values)
    val_encodings[labelCol] = torch.tensor(pd.Series(val[labelCol]).values)
    test_encodings[labelCol] = torch.tensor(pd.Series(test[labelCol]).values)

    return train_encodings, val_encodings, test_encodings


In [14]:
class HateSpeechDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

## Training

In [15]:
def load_model(df, model_path):
    num_labels = len(set(df['labels']))
    model = BertForSequenceClassification.from_pretrained(model_path, num_labels=num_labels)
    # "bert-base-uncased"
    return model.to(device)

In [16]:
class CustomTrainer(Trainer):
    def __init__(self, *args, loss_fn=None, **kwargs):
      super().__init__(*args, **kwargs)
      self.loss_fn = loss_fn

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")  # Get the true labels
        outputs = model(**inputs)  # Get model outputs
        logits = outputs.get("logits")  # Get the logits (raw predictions)

        # Compute the weighted loss
        loss = self.loss_fn(logits, labels)

        # Return loss, and optionally, the outputs (for debugging/metrics)
        return (loss, outputs) if return_outputs else loss

In [21]:
class model_training:
    def __init__(self, train_dataset=None, val_dataset=None, tokenizer=None, model=None, loss_fn=None,
                 lr=None, weight_decay = None, epochs=None, batch_size=None):
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.tokenizer = tokenizer
        self.model = model
        self.loss_fn = loss_fn

        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size

    @staticmethod
    def compute_metrics(p):
        preds = p.predictions.argmax(-1)  # Get predicted labels
        labels = p.label_ids  # True labels

        # Calculate accuracy
        accuracy = accuracy_score(labels, preds)

        # Calculate precision, recall, and F1 score
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        }

    def main(self):
        training_args = TrainingArguments(
            output_dir="./results",
            evaluation_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            save_total_limit=2,
            learning_rate=self.lr,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            num_train_epochs=self.epochs,
            weight_decay=self.weight_decay,
            remove_unused_columns=False,
            load_best_model_at_end=True,
            report_to="none"  # Disables wandb logging
        )

        # Look for the latest checkpoint
        checkpoint_dir = training_args.output_dir
        last_checkpoint = None

        if os.path.isdir(checkpoint_dir):
            checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
            if checkpoints:
                last_checkpoint = os.path.join(checkpoint_dir, sorted(checkpoints, key=lambda x: int(x.split('-')[-1]))[-1])
        if last_checkpoint:
            print(f"Resuming training from checkpoint: {last_checkpoint}")
        else:
            print("No checkpoint found, starting from scratch.")

        if self.loss_fn != None:
            trainer = CustomTrainer(
                model=self.model,
                args=training_args,
                train_dataset=self.train_dataset,
                eval_dataset=self.val_dataset,
                tokenizer=self.tokenizer,
                compute_metrics=self.compute_metrics,
                loss_fn=self.loss_fn
            )
        else:
            trainer = Trainer(
              model=self.model,
              args=training_args,
              train_dataset=self.train_dataset,
              eval_dataset=self.val_dataset,
              tokenizer=self.tokenizer,
              compute_metrics=self.compute_metrics
            )

        return trainer, last_checkpoint

In [18]:
class DatasetPipeline():
    def __init__(self, lang, train_path, val_path, test_path, text_col, label_col, split_again,
                 cols_to_drop, classes, model_path, original, lr=5e-5, weight_decay=0.01, epochs=2, batch_size=16):
        self.lang = lang
        self.train_path = train_path
        self.val_path = val_path
        self.test_path = test_path
        self.text_col = text_col
        self.label_col = label_col
        self.split_again = split_again
        self.cols_to_drop = cols_to_drop
        self.classes = classes
        self.model_path = model_path

        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.original = original

    def prepare_dataset(self, train, val, test):

        # Change the labels of the Bulgarian dataset so that 0 = non-hateful and 1 = hateful
        if self.lang == "bg":
            train.loc[train[self.label_col] >= 1, self.label_col] = 1
            val.loc[val[self.label_col] >= 1, self.label_col] = 1
            test.loc[test[self.label_col] >= 1, self.label_col] = 1

        # Split the dataset into train (80%), validation (10%) and test (10%)
        if self.split_again == True:
            # Merge the current train, test and validation splits into a single dataframe
            merged = pd.concat([train, val, test], axis=0, ignore_index=True)
            # Split
            train, df_temp = train_test_split(merged, test_size=0.2, random_state=42)
            val, test = train_test_split(df_temp, test_size=0.5, random_state=42)

        # Removed failed translations
        train, val, test = remove_failed_translations(train, val, test, self.text_col)

        # Remove unnecessary columns
        train, val, test = drop_columns(train, val, test, self.cols_to_drop, self.label_col)

        # Replace string labels with numbers
        if self.lang == "nl":
            sentiment_map = {'non-hateful': 0, 'hateful': 1}
            train['labels'] = train['labels'].map(sentiment_map)
            val['labels'] = val['labels'].map(sentiment_map)
            test['labels'] = test['labels'].map(sentiment_map)

        if self.lang == "ru":
            train['labels'] = train['labels'].astype(int)
            val['labels'] = val['labels'].astype(int)
            test['labels'] = test['labels'].astype(int)

        return train, val, test

    def preprocess_dataset(self, train, val, test):

        # Compute the weighted loss function
        loss_fn = compute_class_weights(train, self.classes, device)

        # Preprocess
        train_cleaned, val_cleaned, test_cleaned = preprocess_dataset(train, val, test, self.text_col, self.original)

        tokenizer = get_tokenizer(self.model_path)

        # Tokenization
        print("Tokenizing...")
        train_encodings, val_encodings, test_encodings = tokenization(train, val, test, 'labels', self.text_col, tokenizer)

        return train_encodings, val_encodings, test_encodings, loss_fn, tokenizer

    def main(self):
        # Load the dataset
        train, val, test = load_dataset_splits(self.train_path, self.val_path, self.test_path)

        # Prepare the dataset
        train, val, test = self.prepare_dataset(train, val, test)

        # Preprocess the dataset
        train_encodings, val_encodings, test_encodings, loss_fn, tokenizer = self.preprocess_dataset(train, val, test)

        # Create the HSD Dataset
        train_dataset = HateSpeechDataset(train_encodings, train_encodings['labels'])
        val_dataset = HateSpeechDataset(val_encodings, val_encodings['labels'])
        test_dataset = HateSpeechDataset(test_encodings, test_encodings['labels'])

        # Load the model
        model = load_model(train, self.model_path)

        # Get trainer
        training = model_training(
                  train_dataset=train_dataset,
                  val_dataset=val_dataset,
                  tokenizer=tokenizer,
                  model=model,
                  loss_fn=loss_fn,
                  lr=self.lr,
                  weight_decay=self.weight_decay,
                  epochs=self.epochs,
                  batch_size=self.batch_size
              )
        trainer, last_checkpoint = training.main()
        # Resume training from the last checkpoint if it exists
        trainer.train(resume_from_checkpoint=last_checkpoint)
        return trainer

## Bulgarian

### Bulgarian Translated

In [ ]:
train_bg_path_translated = "/content/HSD_ch_train_translated.csv"
val_bg_path_translated = "/content/HSD_ch_val_translated.csv"
test_bg_path_translated = "/content/HSD_ch_test_translated.csv"

dataset_pipeline_bg_translated = DatasetPipeline(lang="bg", train_path=train_bg_path_translated, val_path=val_bg_path_translated, test_path=test_bg_path_translated,
                text_col="translated_text", label_col="Type", split_again=False,
                cols_to_drop=["text", "Positivity"], classes=np.array([0, 1]),
                model_path="bert-base-uncased", original=False)

In [ ]:
trainer_bg_translated = dataset_pipeline_bg_translated.main()

### Bulgarian original

In [ ]:
train_bg_path_original = "/content/HSD_ch_train.csv"
val_bg_path_original = "/content/HSD_ch_val.csv"
test_bg_path_original = "/content/HSD_ch_test.csv"

dataset_pipeline_bg_original = DatasetPipeline(lang="bg", train_path=train_bg_path_original, val_path=val_bg_path_original, test_path=test_bg_path_original,
                text_col="text", label_col="Type", split_again=True,
                cols_to_drop=["Positivity"], classes=np.array([0, 1]),
                model_path="AIaLT-IICT/bert_bg_lit_web_base_uncased", original=True)

## Chinese

### Chinese Translated

In [22]:
train_ch_path_translated = "/content/HSD_ch_train_translated.csv"
val_ch_path_translated = "/content/HSD_ch_val_translated.csv"
test_ch_path_translated = "/content/HSD_ch_test_translated.csv"

dataset_pipeline_ch_translated = DatasetPipeline(lang="ch", train_path=train_ch_path_translated, val_path=val_ch_path_translated, test_path=test_ch_path_translated,
                text_col="translated_TEXT", label_col="label", split_again=False,
                cols_to_drop=["Unnamed: 0", "split", "topic", "TEXT"], classes=np.array([0, 1]),
                model_path="bert-base-uncased", original=False)

In [23]:
trainer_ch_translated = dataset_pipeline_ch_translated.main()

Nan values found in train set:  0
Nan values found in validation set:  0
Nan values found in test set:  0
Preprocessing...


<ipython-input-11-f242e94a4706>:8: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags


Tokenizing...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-16-7502dd44e21a>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


No checkpoint found, starting from scratch.


<ipython-input-14-d47c96fff4a1>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

### Chinese original

In [ ]:
train_ch_path_original = "/content/train.csv"
val_ch_path_original = "/content/dev.csv"
test_ch_path_original = "/content/test.csv"

dataset_pipeline_ch_original = DatasetPipeline(lang="ch", train_path=train_ch_path_original, val_path=val_ch_path_original, test_path=test_ch_path_original,
                text_col="TEXT", label_col="label", split_again=True,
                cols_to_drop=["split", "topic"], classes=np.array([0, 1]),
                model_path="google-bert/bert-base-chinese", original=True)

## Dutch

### Dutch Translated

In [ ]:
train_nl_path_translated = "/content/HSD_nl_train_translated.csv"
val_nl_path_translated = "/content/HSD_nl_val_translated.csv"
test_nl_path_translated = "/content/HSD_nl_test_translated.csv"

dataset_pipeline_nl_translated = DatasetPipeline(lang="nl", train_path=train_nl_path_translated, val_path=val_nl_path_translated, test_path=test_nl_path_translated,
                text_col="translated_test_case", label_col="label_gold", split_again=False,
                cols_to_drop=["test_case"], classes=np.array([0, 1]),
                model_path="bert-base-uncased", original=False)

In [ ]:
trainer_nl_translated = dataset_pipeline_nl_translated.main()

### Dutch original

In [ ]:
train_nl_path_original = "/content/HSD_nl_train.csv"
val_nl_path_original = "/content/HSD_nl_val.csv"
test_nl_path_original = "/content/HSD_nl_test.csv"

dataset_pipeline_nl_original = DatasetPipeline(lang="nl", train_path=train_nl_path_original, val_path=val_nl_path_original, test_path=test_nl_path_original,
                text_col="test_case", label_col="label_gold", split_again=False,
                cols_to_drop=[""], classes=np.array([0, 1]),
                model_path="GroNLP/bert-base-dutch-cased", original=True)

## Italian

### Italian Translated

In [ ]:
train_it_path_translated = "/content/HSD_it_train_translated.csv"
val_it_path_translated = "/content/HSD_it_val_translated.csv"
test_it_path_translated = "/content/HSD_it_test_translated.csv"

dataset_pipeline_it_translated = DatasetPipeline(lang="it", train_path=train_it_path_translated, val_path=val_it_path_translated, test_path=test_it_path_translated,
                text_col="translated_full_text", label_col="hs", split_again=False,
                cols_to_drop=["id", "full_text", "stereotype"], classes=np.array([0, 1]),
                model_path="bert-base-uncased", original=False)

In [ ]:
trainer_it_translated = dataset_pipeline_it_translated.main()

### Italian Original

In [ ]:
train_it_path_original = "/content/HSD_it_train_translated.csv"
val_it_path_original = "/content/HSD_it_val_translated.csv"
test_it_path_original = "/content/HSD_it_test_translated.csv"

dataset_pipeline_it_original = DatasetPipeline(lang="it", train_path=train_it_path_original, val_path=val_it_path_original, test_path=test_it_path_original,
                text_col="full_text", label_col="hs", split_again=False,
                cols_to_drop=["id", "translated_full_text", "stereotype"], classes=np.array([0, 1]),
                model_path="dbmdz/bert-base-italian-uncased", original=True)

In [ ]:
trainer_it_original = dataset_pipeline_it_original.main()

## Russian

### Russian Translated

In [ ]:
train_ru_path_translated = "/content/HSD_ru_train_translated.csv"
val_ru_path_translated = "/content/HSD_ru_val_translated.csv"
test_ru_path_translated = "/content/HSD_ru_test_translated.csv"

dataset_pipeline_ru_translated = DatasetPipeline(lang="ru", train_path=train_ru_path_translated, val_path=val_ru_path_translated, test_path=test_ru_path_translated,
                text_col="translated_text", label_col="hate_speech", split_again=False,
                cols_to_drop=["Unnamed: 0", "index", "text", "sentiment"], classes=np.array([0, 1]),
                model_path="bert-base-uncased", original=False)

In [ ]:
trainer_ru_translated = dataset_pipeline_ru_translated.main()

### Russian Original

In [ ]:
train_ru_path_original = "/content/HSD_ru_train_translated.csv"
val_ru_path_original = "/content/HSD_ru_val_translated.csv"
test_ru_path_original = "/content/HSD_ru_test_translated.csv"

dataset_pipeline_ru_original = DatasetPipeline(lang="ru", train_path=train_ru_path_original, val_path=val_ru_path_original, test_path=test_ru_path_original,
                text_col="text", label_col="hate_speech", split_again=False,
                cols_to_drop=["Unnamed: 0", "index", "translated_text", "sentiment"], classes=np.array([0, 1]),
                model_path="deepvk/bert-base-uncased", original=True)

In [ ]:
trainer_ru_original = dataset_pipeline_ru_original.main()